## 1. Install & Import Library

In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install wandb

from unsloth import FastLanguageModel
from google.colab import userdata
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from trl import SFTTrainer, SFTConfig
import torch
import os
import wandb

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-zok2ryva/unsloth_47ce7341e5924d44962a58f46bfbd0ff
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-zok2ryva/unsloth_47ce7341e5924d44962a58f46bfbd0ff
  Resolved https://github.com/unslothai/unsloth.git to commit 677ec0cc20bf7cb4735385c51a22999a64839a83
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 107.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 22.5 MB/s eta 0:00:00
   

## 2. Mount Google Drive

Dipakai buat nyimpen checkpoint training, biar kalau Colab disconnect di tengah jalan, kita bisa **resume** dari checkpoint terakhir tanpa training ulang dari nol.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_BASE_DIR = "/content/drive/MyDrive/chatbot-rag-hukum/checkpoints"
os.makedirs(CHECKPOINT_BASE_DIR, exist_ok=True)
print(f"📁 Checkpoint akan disimpan di: {CHECKPOINT_BASE_DIR}")


Mounted at /content/drive
📁 Checkpoint akan disimpan di: /content/drive/MyDrive/chatbot-rag-hukum/checkpoints


## 3. Login ke Weights & Biases

Simpan API key W&B di Colab Secrets dengan nama `WANDB_API_KEY` (ambil dari https://wandb.ai/authorize).

In [4]:
wandb_key = userdata.get("WANDB_API_KEY")
wandb.login(key=wandb_key)


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ganggaswara11 (ganggaswara11-universitas-pendidikan-ganesha) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 4. Load Base Model (Llama 3.2 3B Instruct, 4-bit)

In [5]:
max_seq_length = 1024
dtype = None
load_in_4bit = True

def load_fresh_model():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    tokenizer = get_chat_template(tokenizer, chat_template = "llama-3")
    return model, tokenizer


## 5. Load & Split Dataset (Train/Validation) — Kriteria Skilled #1

In [6]:
raw_dataset = load_dataset("Ichsan2895/alpaca-gpt4-indonesian", split="train")

split_dataset = raw_dataset.train_test_split(test_size=0.1, seed=3407)
train_dataset_raw = split_dataset["train"]
eval_dataset_raw = split_dataset["test"]

print(f"Jumlah data train     : {len(train_dataset_raw)}")
print(f"Jumlah data validation: {len(eval_dataset_raw)}")


README.md:   0%|          | 0.00/1.91k [00:00<?, ?B/s]

alpaca-gpt4-indonesia.csv:   0%|          | 0.00/41.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49969 [00:00<?, ? examples/s]

Jumlah data train     : 44972
Jumlah data validation: 4997


## 6. Format Dataset dengan Chat Template Llama-3

In [7]:
def formatting_prompts_func(examples, tokenizer):
    inputs  = examples["input"]
    outputs = examples["output"]
    texts = []
    for input_text, output_text in zip(inputs, outputs):
        messages = [
            {"role": "system",    "content": "Kamu adalah asisten AI yang membantu, akurat, dan jujur. Jawab dalam Bahasa Indonesia."},
            {"role": "user",      "content": input_text.strip()},
            {"role": "assistant", "content": output_text.strip()},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

In [8]:
# ── Preview Dataset Setelah Mapping Chat Template ──────────────────────────
_, _tok_preview = load_fresh_model()

_preview_mapped = train_dataset_raw.select(range(3)).map(
    lambda ex: formatting_prompts_func(ex, _tok_preview), batched=True
)

print("="*70)
print("PREVIEW DATASET SETELAH MAPPING CHAT TEMPLATE (3 sampel pertama)")
print("="*70)
for idx, sample in enumerate(_preview_mapped):
    print(f"\n--- Sampel [{idx+1}] ---")
    print(sample["text"])
    print()

del _tok_preview
import gc, torch; gc.collect(); torch.cuda.empty_cache()
print("Preview selesai. Memori GPU sudah dibersihkan.")

==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

PREVIEW DATASET SETELAH MAPPING CHAT TEMPLATE (3 sampel pertama)

--- Sampel [1] ---
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Kamu adalah asisten AI yang membantu, akurat, dan jujur. Jawab dalam Bahasa Indonesia.<|eot_id|><|start_header_id|>user<|end_header_id|>

Buat kategori baru untuk buku resep.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"Makanan dalam Waktu Kurang dari 20 Menit: Resep Cepat dan Segar untuk Koki Sibuk"<|eot_id|>


--- Sampel [2] ---
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Kamu adalah asisten AI yang membantu, akurat, dan jujur. Jawab dalam Bahasa Indonesia.<|eot_id|><|start_header_id|>user<|end_header_id|>

Tulis naskah percakapan antara penjaga toko dan pelanggan yang perlu menukar produk.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Penjaga Toko: Selamat pagi, ada yang bisa saya bantu hari ini?

Pelanggan: Hai, saya membeli produk ini beberapa hari lalu, tetapi tidak berfungsi seperti yang diharapka

## 7. Fungsi `run_experiment` — dengan Checkpoint + Auto-Resume dari Google Drive

Sebelum training mulai, fungsi ini **cek dulu** apakah ada checkpoint lama di Google Drive untuk `exp_name` yang sama. Kalau ada, training otomatis **lanjut dari checkpoint terakhir**, bukan dari step 0. Kalau Colab disconnect di tengah jalan, kamu tinggal jalankan ulang cell eksperimennya — gak perlu mulai dari awal lagi.

In [9]:
def run_experiment(exp_name, learning_rate, lora_r, lora_alpha, max_steps=800):
    print(f"\n{'='*60}\n🧪 Menjalankan Eksperimen: {exp_name}\n{'='*60}")

    checkpoint_dir = f"{CHECKPOINT_BASE_DIR}/{exp_name}"

    # --- Cek apakah ada checkpoint lama di Drive ---
    resume_from = None
    if os.path.exists(checkpoint_dir):
        checkpoints = [d for d in os.listdir(checkpoint_dir) if d.startswith("checkpoint-")]
        if checkpoints:
            latest = max(checkpoints, key=lambda x: int(x.split("-")[1]))
            resume_from = os.path.join(checkpoint_dir, latest)
            print(f"📂 Checkpoint ditemukan! Melanjutkan training dari: {resume_from}")
        else:
            print("📂 Folder checkpoint ada tapi kosong, training dari awal.")
    else:
        print("📂 Belum ada checkpoint, training dari awal (step 0).")

    model, tokenizer = load_fresh_model()

    model = FastLanguageModel.get_peft_model(
        model,
        r = lora_r,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj",],
        lora_alpha = lora_alpha,
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
        use_rslora = False,
        loftq_config = None,
    )

    mapped_train = train_dataset_raw.map(
        lambda ex: formatting_prompts_func(ex, tokenizer), batched=True
    )
    mapped_eval = eval_dataset_raw.map(
        lambda ex: formatting_prompts_func(ex, tokenizer), batched=True
    )

    sft_config = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        max_steps = max_steps,
        learning_rate = learning_rate,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = checkpoint_dir,
        save_strategy = "steps",
        save_steps = 50,
        save_total_limit = 2,        # cuma simpan 2 checkpoint terbaru biar Drive gak penuh
        eval_strategy = "steps",
        eval_steps = 50,
        report_to = "wandb",
        run_name = exp_name,
    )

    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = mapped_train,
        eval_dataset = mapped_eval,
        dataset_text_field = "text",
        max_seq_length = 1024,
        dataset_num_proc = 2,
        packing = False,
        args = sft_config,
    )

    trainer_stats = trainer.train(resume_from_checkpoint=resume_from)

    eval_losses = [log["eval_loss"] for log in trainer.state.log_history if "eval_loss" in log]
    final_eval_loss = eval_losses[-1] if eval_losses else None

    print(f"✅ {exp_name} selesai. Final eval_loss: {final_eval_loss:.4f}")
    wandb.finish()

    return {
        "exp_name": exp_name,
        "model": model,
        "tokenizer": tokenizer,
        "eval_loss": final_eval_loss,
        "config": {"learning_rate": learning_rate, "lora_r": lora_r, "lora_alpha": lora_alpha},
    }

print("✅ Fungsi run_experiment siap dipanggil.")


✅ Fungsi run_experiment siap dipanggil.


## 8. Eksperimen A

`learning_rate=2e-4`, LoRA `r=16`.

Kalau Colab disconnect di tengah jalan, cukup jalankan ulang cell ini — otomatis lanjut dari checkpoint terakhir di Drive.

In [10]:
result_a = run_experiment("exp_A_lr2e4_r16", learning_rate=2e-4, lora_r=16, lora_alpha=16, max_steps=800)



🧪 Menjalankan Eksperimen: exp_A_lr2e4_r16
📂 Checkpoint ditemukan! Melanjutkan training dari: /content/drive/MyDrive/chatbot-rag-hukum/checkpoints/exp_A_lr2e4_r16/checkpoint-800
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.6.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Map:   0%|          | 0/44972 [00:00<?, ? examples/s]

Map:   0%|          | 0/4997 [00:00<?, ? examples/s]

...

Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
801,1.095258,1.076411


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/chatbot-rag-hukum/checkpoints/exp_A_lr2e4_r16/checkpoint-801/tokenizer_config.json.


✅ exp_A_lr2e4_r16 selesai. Final eval_loss: 1.0764


eval/loss,▁
eval/runtime,▁
eval/samples_per_second,▁
eval/steps_per_second,▁
train/epoch,▁▁
train/global_step,▁▁
eval/loss,1.07641
eval/runtime,632.023
eval/samples_per_second,7.906
eval/steps_per_second,1.978
total_flos,3.0331432055310336e+16


## 9. Eksperimen B

`learning_rate=1e-4`, LoRA `r=32`.

Dipisah dari Eksperimen A biar kalau salah satu gagal, yang lain gak perlu diulang.

In [11]:
result_b = run_experiment("exp_B_lr1e4_r32", learning_rate=1e-4, lora_r=32, lora_alpha=32, max_steps=800)



🧪 Menjalankan Eksperimen: exp_B_lr1e4_r32
📂 Checkpoint ditemukan! Melanjutkan training dari: /content/drive/MyDrive/chatbot-rag-hukum/checkpoints/exp_B_lr1e4_r32/checkpoint-800
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


Map:   0%|          | 0/44972 [00:00<?, ? examples/s]

Map:   0%|          | 0/4997 [00:00<?, ? examples/s]

...

Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
801,1.101457,1.077274


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

✅ exp_B_lr1e4_r32 selesai. Final eval_loss: 1.0773


eval/loss,▁
eval/runtime,▁
eval/samples_per_second,▁
eval/steps_per_second,▁
train/epoch,▁▁
train/global_step,▁▁
eval/loss,1.07727
eval/runtime,1014.9217
eval/samples_per_second,4.924
eval/steps_per_second,1.232
total_flos,3.059082642178253e+16


## 10. Bandingkan Hasil & Pilih Model Terbaik

In [12]:
print("📊 Perbandingan hasil eksperimen:")
print(f"  {result_a['exp_name']} -> eval_loss: {result_a['eval_loss']:.4f} | config: {result_a['config']}")
print(f"  {result_b['exp_name']} -> eval_loss: {result_b['eval_loss']:.4f} | config: {result_b['config']}")

best_result = result_a if result_a["eval_loss"] < result_b["eval_loss"] else result_b
print(f"\n🏆 Eksperimen terbaik: {best_result['exp_name']} (eval_loss paling kecil: {best_result['eval_loss']:.4f})")

best_model = best_result["model"]
best_tokenizer = best_result["tokenizer"]


📊 Perbandingan hasil eksperimen:
  exp_A_lr2e4_r16 -> eval_loss: 1.0764 | config: {'learning_rate': 0.0002, 'lora_r': 16, 'lora_alpha': 16}
  exp_B_lr1e4_r32 -> eval_loss: 1.0773 | config: {'learning_rate': 0.0001, 'lora_r': 32, 'lora_alpha': 32}

🏆 Eksperimen terbaik: exp_A_lr2e4_r16 (eval_loss paling kecil: 1.0764)


## 11. Push Model Terbaik ke HuggingFace Hub

In [13]:
hf_token = userdata.get("HF_TOKEN")
model_id = "Ganggas/legal-chatbot-llama3-indo-3b-final"

best_model.config._name_or_path = "unsloth/Llama-3.2-3B-Instruct"

print(f"⏳ Mengunggah model terbaik ({best_result['exp_name']}) ke {model_id}...")
best_model.push_to_hub_merged(
    model_id,
    best_tokenizer,
    save_method = "merged_16bit",
    token = hf_token
)
print("✅ Upload berhasil! Pastikan cek di huggingface.co bahwa repo berstatus PUBLIC.")


⏳ Mengunggah model terbaik (exp_A_lr2e4_r16) ke Ganggas/legal-chatbot-llama3-indo-3b-final...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in Ganggas/legal-chatbot-llama3-indo-3b-final/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00002.safetensors:   0%|          |  608kB / 1.46GB            


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [04:26<00:00, 133.34s/it]


Unsloth: Merge process complete. Saved to `/content/Ganggas/legal-chatbot-llama3-indo-3b-final`
✅ Upload berhasil! Pastikan cek di huggingface.co bahwa repo berstatus PUBLIC.
